# Final OCSD: a new, natural example scene for the module slides (M1-M5)

Scene **qd_3_simple_003** ("a tall tree and two gray birds by a lake"), seed 0, with the **final OCSD**: SD 1.5, all five modules M1-M5, M3 on, harmonised M4 frozen in `results/m4_study_v1/locked.json`.
Saves (a) every module's intermediate images (M1 masks and boxes, M2 candidates and scores, M3 identity check, M4 steps 1/6/12/18/24/30, M5 attention and final check) and (b) the same sketch with 3 other backgrounds.

**How to run:** Runtime -> Change runtime type -> any GPU (T4, L4 or A100) -> **Run all**. About 20 minutes. One ZIP downloads at the end; it is also on Drive at `KLTN_OCSD/trace_ocsd/EXAMPLE_qd_3_simple_003_s0.zip`.


In [ ]:
#@title 1. Settings
SID    = "qd_3_simple_003"   #@param {type:"string"}
SEED   = 0                   #@param {type:"integer"}
BGS    = ["in a city park", "on a snowy field", "on a sandy beach"]
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"  #@param {type:"string"}


In [ ]:
#@title 2. Drive, GPU check, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode or int(gpu.stdout.strip().split(",")[-1]) < 14000:
    raise RuntimeError("This needs a GPU with at least 14 GB (T4, L4 or A100): Runtime > Change runtime type > GPU, then Run all")
# OCSD on SD 1.5 (with M3 training) fits in about 12 GB, so a T4, L4 or A100 all work
CODE = "/content/KLTN_OCSD_example"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])

In [ ]:
#@title 3. Final OCSD: steps, M2 scores, attention + own background + 3 others (about 10 minutes after the models are downloaded)
from ocsd import bg_swap
FINAL = bg_swap.final_overrides(CODE)
print("final OCSD settings:", FINAL)
META = bg_swap.run(DRIVE_ROOT, sid=SID, seed=SEED, bgs=BGS, overrides=FINAL, tag="_final", record_steps=(0, 5, 11, 17, 23, 29), attn_steps=(11, 23))
for r in META["runs"]: print(r["bg"], "|", r["global_prompt"], "|", r["seconds"], "s, tries", r["log"].get("tries"))
print("M4 anchor weights:", META["steps"]["m4"])


In [ ]:
#@title 4. Same scene through the module tracer (M1 masks/boxes, M3 identity check, M5 final check), final OCSD settings
from ocsd import trace_ocsd
import gc, torch
gc.collect(); torch.cuda.empty_cache()
TOUT = os.path.join(DRIVE_ROOT, "trace_ocsd", "final")
TMETA = trace_ocsd.trace(DRIVE_ROOT, sid=SID, seed=SEED, overrides=FINAL, out_root=TOUT)
print("M3 check rows:", len(TMETA["m3"]["check"]), "| M5(d) tries:", TMETA["m5d"]["tries"])


In [ ]:
#@title 5. One ZIP with everything, then download
import zipfile
from IPython.display import Image, display
b_out = os.path.join(DRIVE_ROOT, "trace_ocsd", f"bgswap_{SID}_s{SEED}_final")
t_out = os.path.join(TOUT, f"{SID}_s{SEED}")
display(Image(os.path.join(b_out, "SHEET.jpg"), width=1400))
zpath = os.path.join(DRIVE_ROOT, "trace_ocsd", f"EXAMPLE_{SID}_s{SEED}.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, name in ((b_out, "bgswap"), (t_out, "trace")):
        for dp, _, fs in os.walk(root):
            for f in fs:
                p = os.path.join(dp, f)
                z.write(p, os.path.join(name, os.path.relpath(p, root)))
print("ZIP:", zpath, os.path.getsize(zpath) // 1024, "KB")
from google.colab import files
try:
    files.download(zpath)
except Exception as exc:
    print("Get it from Drive:", zpath, exc)
print("Done. Disconnect the GPU runtime.")
